# 01 Data Download and Feature Engineering

This notebook prepares the data used throughout the project.

It loads or refreshes raw market data, cleans the equity and proxy panels, constructs the forecasting target, engineers the feature set, and saves a reusable modeling table for the downstream notebooks.

In [1]:
from pathlib import Path
import json
import time
import warnings

import numpy as np
import pandas as pd
import yfinance as yf

warnings.filterwarnings('ignore', category=FutureWarning)
warnings.filterwarnings('ignore', category=Warning, module='yfinance')

PROJECT_DIR = Path('/Users/chonggu/Documents/Mine/Erdos 2026 Spring final project')
DATA_DIR = PROJECT_DIR / 'data'
DATA_DIR.mkdir(parents=True, exist_ok=True)

CORE_DATA_PATH = DATA_DIR / 'core_universe_ohlcv_2016_2025.csv'
MARKET_PROXY_PATH = DATA_DIR / 'market_proxies_ohlcv_2016_2025.csv'
MACRO_PROXY_PATH = DATA_DIR / 'macro_credit_proxies_ohlcv_2016_2025.csv'
MANIFEST_PATH = DATA_DIR / 'download_manifest.json'
MODEL_DF_PATH = DATA_DIR / 'model_df_2016_2025.csv.gz'

CORE_TICKERS = [
    'AAPL', 'MSFT', 'GOOGL', 'AMZN', 'META', 'NVDA', 'TSLA', 'ADBE', 'CRM', 'ORCL',
    'JPM', 'BAC', 'GS', 'MS', 'WFC', 'C', 'BLK', 'AXP', 'SCHW', 'USB',
    'JNJ', 'PFE', 'MRK', 'LLY', 'ABBV', 'TMO', 'UNH', 'CVS', 'AMGN', 'GILD',
    'KO', 'PEP', 'WMT', 'COST', 'HD', 'MCD', 'NKE', 'SBUX', 'TGT', 'DIS',
    'XOM', 'CVX', 'COP', 'SLB', 'EOG', 'BA', 'CAT', 'GE', 'HON', 'UPS',
    'SPY', 'QQQ', 'IWM', 'DIA', 'VTI',
]
MARKET_PROXIES = ['SPY', 'QQQ', 'IWM', 'VTI']
MACRO_CREDIT_PROXIES = ['^VIX', '^TNX', '^IRX', 'HYG', 'LQD']
OHLCV_COLUMNS = ['Open', 'High', 'Low', 'Close', 'Adj Close', 'Volume']

START_DATE = '2016-01-01'
END_DATE = '2026-01-01'
REFRESH_RAW_DATA = False

PROJECT_DIR

PosixPath('/Users/chonggu/Documents/Mine/Erdos 2026 Spring final project')

## Optional raw-data refresh

Set `REFRESH_RAW_DATA = True` only when the raw Yahoo Finance files need to be refreshed. Otherwise the notebook reuses the saved CSV files in `data/` for reproducibility.

In [2]:
def normalize_download(df: pd.DataFrame) -> pd.DataFrame:
    if df.empty:
        raise ValueError('yfinance returned an empty DataFrame.')

    df = df.copy()
    df.index.name = 'Date'

    if isinstance(df.columns, pd.MultiIndex):
        long_df = df.stack(level=0).reset_index().rename(columns={'level_1': 'Ticker'})
    else:
        long_df = df.reset_index()
        long_df['Ticker'] = ''

    for column in OHLCV_COLUMNS:
        if column not in long_df.columns:
            long_df[column] = pd.NA

    ordered = ['Date', 'Ticker'] + OHLCV_COLUMNS
    long_df = long_df[ordered].sort_values(['Ticker', 'Date']).reset_index(drop=True)
    long_df['Date'] = pd.to_datetime(long_df['Date']).dt.strftime('%Y-%m-%d')
    return long_df

def download_one_ticker(ticker: str, start: str, end: str, retries: int = 4) -> pd.DataFrame:
    last_error = None
    for attempt in range(1, retries + 1):
        try:
            data = yf.download(
                tickers=ticker,
                start=start,
                end=end,
                auto_adjust=False,
                actions=False,
                progress=False,
                threads=False,
                group_by='ticker',
            )
            if data.empty:
                raise ValueError(f'No data returned for {ticker}.')

            normalized = normalize_download(data)
            if normalized['Ticker'].isna().all() or (normalized['Ticker'] == '').all():
                normalized['Ticker'] = ticker
            return normalized
        except Exception as exc:
            last_error = exc
            time.sleep(5 * attempt)
    raise RuntimeError(f'Failed to download {ticker}') from last_error

def download_ohlcv(tickers: list[str], start: str, end: str) -> pd.DataFrame:
    frames = []
    for ticker in tickers:
        frames.append(download_one_ticker(ticker, start, end))
        time.sleep(2)
    combined = pd.concat(frames, ignore_index=True)
    return combined.sort_values(['Ticker', 'Date']).reset_index(drop=True)

def build_manifest(datasets: dict[str, pd.DataFrame], start: str, end: str) -> dict:
    manifest = {'start': start, 'end_exclusive': end, 'datasets': {}}
    for name, df in datasets.items():
        manifest['datasets'][name] = {
            'rows': int(len(df)),
            'tickers': sorted(df['Ticker'].dropna().unique().tolist()),
            'min_date': None if df.empty else df['Date'].min(),
            'max_date': None if df.empty else df['Date'].max(),
        }
    return manifest


In [3]:
if REFRESH_RAW_DATA:
    core_download_df = download_ohlcv(CORE_TICKERS, START_DATE, END_DATE)
    market_download_df = core_download_df[core_download_df['Ticker'].isin(MARKET_PROXIES)].reset_index(drop=True)
    macro_download_df = download_ohlcv(MACRO_CREDIT_PROXIES, START_DATE, END_DATE)

    core_download_df.to_csv(CORE_DATA_PATH, index=False)
    market_download_df.to_csv(MARKET_PROXY_PATH, index=False)
    macro_download_df.to_csv(MACRO_PROXY_PATH, index=False)

    manifest = build_manifest(
        {
            'core_universe': core_download_df,
            'market_proxies': market_download_df,
            'macro_credit_proxies': macro_download_df,
        },
        START_DATE,
        END_DATE,
    )
    MANIFEST_PATH.write_text(json.dumps(manifest, indent=2))
    print('Raw data refreshed and saved.')
else:
    print('REFRESH_RAW_DATA is False. Reusing existing raw CSV files from data/.')

sorted([path.name for path in DATA_DIR.iterdir()])

REFRESH_RAW_DATA is False. Reusing existing raw CSV files from data/.


['core_universe_ohlcv_2016_2025.csv',
 'core_universe_tickers.txt',
 'download_manifest.json',
 'macro_credit_proxies_ohlcv_2016_2025.csv',
 'market_proxies_ohlcv_2016_2025.csv',
 'proxy_tickers.txt']

## Proxy meaning

- `SPY`: S&P 500 ETF, broad U.S. large-cap market proxy.
- `QQQ`: Nasdaq-100 ETF, growth and technology-heavy market proxy.
- `IWM`: Russell 2000 ETF, small-cap proxy.
- `VTI`: total U.S. stock market ETF.
- `^VIX`: implied volatility / market stress proxy.
- `^TNX`: 10-year Treasury yield proxy.
- `^IRX`: short-rate proxy.
- `HYG`: high-yield credit proxy.
- `LQD`: investment-grade credit proxy.

In [4]:
with open(MANIFEST_PATH) as f:
    manifest = json.load(f)

manifest

{'downloaded_at_utc': '2026-03-17T18:50:07.030914+00:00',
 'start': '2016-01-01',
 'end_exclusive': '2026-01-01',
 'datasets': {'core_universe': {'rows': 138270,
   'tickers': ['AAPL',
    'ABBV',
    'ADBE',
    'AMGN',
    'AMZN',
    'AXP',
    'BA',
    'BAC',
    'BLK',
    'C',
    'CAT',
    'COP',
    'COST',
    'CRM',
    'CVS',
    'CVX',
    'DIA',
    'DIS',
    'EOG',
    'GE',
    'GILD',
    'GOOGL',
    'GS',
    'HD',
    'HON',
    'IWM',
    'JNJ',
    'JPM',
    'KO',
    'LLY',
    'MCD',
    'META',
    'MRK',
    'MS',
    'MSFT',
    'NKE',
    'NVDA',
    'ORCL',
    'PEP',
    'PFE',
    'QQQ',
    'SBUX',
    'SCHW',
    'SLB',
    'SPY',
    'TGT',
    'TMO',
    'TSLA',
    'UNH',
    'UPS',
    'USB',
    'VTI',
    'WFC',
    'WMT',
    'XOM'],
   'min_date': '2016-01-04',
   'max_date': '2025-12-31'},
  'market_proxies': {'rows': 10056,
   'tickers': ['IWM', 'QQQ', 'SPY', 'VTI'],
   'min_date': '2016-01-04',
   'max_date': '2025-12-31'},
  'macro_credit

## Load downloaded data

In [5]:
# Load the raw CSVs exactly as they were downloaded.
raw_core_df = pd.read_csv(CORE_DATA_PATH, parse_dates=['Date'])
raw_market_proxy_df = pd.read_csv(MARKET_PROXY_PATH, parse_dates=['Date'])
raw_macro_proxy_df = pd.read_csv(MACRO_PROXY_PATH, parse_dates=['Date'])


def clean_ohlcv_frame(df, allow_nonpositive_prices=False, fill_missing_volume_with_zero=False):
    """
    Standardize dtypes, remove duplicated (Date, Ticker) rows, sort the panel,
    and handle obviously invalid values.

    For equities / ETFs we require strictly positive price columns.
    For rate-style series such as ^TNX and ^IRX we allow non-positive levels,
    because they are quoted yields rather than stock prices.
    """
    df = df.copy()
    df = df.dropna(subset=['Date', 'Ticker'])
    df = df.drop_duplicates(subset=['Date', 'Ticker'], keep='last')

    numeric_cols = ['Open', 'High', 'Low', 'Close', 'Adj Close', 'Volume']
    for col in numeric_cols:
        df[col] = pd.to_numeric(df[col], errors='coerce')

    df['Date'] = pd.to_datetime(df['Date'])
    df = df.sort_values(['Ticker', 'Date']).reset_index(drop=True)

    if allow_nonpositive_prices:
        # Keep rate-style series even if a quoted level goes slightly below zero.
        df = df.dropna(subset=['Open', 'High', 'Low', 'Close', 'Adj Close'])
    else:
        for col in ['Open', 'High', 'Low', 'Close', 'Adj Close']:
            df.loc[df[col] <= 0, col] = np.nan
        df = df.dropna(subset=['Open', 'High', 'Low', 'Close', 'Adj Close'])

    if fill_missing_volume_with_zero:
        df['Volume'] = df['Volume'].fillna(0).clip(lower=0)
    else:
        df.loc[df['Volume'] < 0, 'Volume'] = np.nan

    return df.reset_index(drop=True)


# Clean the main equity / ETF universe and the proxy tables separately.
core_df = clean_ohlcv_frame(raw_core_df, allow_nonpositive_prices=False)
market_proxy_df = clean_ohlcv_frame(raw_market_proxy_df, allow_nonpositive_prices=False)
macro_proxy_df = clean_ohlcv_frame(
    raw_macro_proxy_df,
    allow_nonpositive_prices=True,
    fill_missing_volume_with_zero=True,
)

cleaning_summary = pd.DataFrame(
    {
        'dataset': ['core', 'market_proxy', 'macro_proxy'],
        'raw_rows': [len(raw_core_df), len(raw_market_proxy_df), len(raw_macro_proxy_df)],
        'clean_rows': [len(core_df), len(market_proxy_df), len(macro_proxy_df)],
        'n_tickers': [core_df['Ticker'].nunique(), market_proxy_df['Ticker'].nunique(), macro_proxy_df['Ticker'].nunique()],
    }
)
cleaning_summary

,dataset,raw_rows,clean_rows,n_tickers
0,core,138270,138270,55
1,market_proxy,10056,10056,4
2,macro_proxy,12568,12568,5


## Define the target: future 5-day realized volatility

For each ticker $i$ and date $t$, define the adjusted-close log return

$$
r_{i,t} = \log\left(\frac{P_{i,t}}{P_{i,t-1}}\right).
$$

The primary target for this project is the annualized future 5-day realized volatility

$$
FVOL^{(5)}_{i,t} = \sqrt{\frac{252}{5}\sum_{j=1}^{5} r_{i,t+j}^2}.
$$

For modeling, we will typically use the log-transformed target `log_fvol_5d`.

In [6]:
def add_basic_price_features(df):
    """
    Build the stock-level features we need for volatility forecasting.

    These features intentionally mix three families of signals:
    1. returns-based signals,
    2. realized / historical volatility signals,
    3. liquidity and intraday range signals.
    """
    df = df.sort_values(['Ticker', 'Date']).copy()
    grouped = df.groupby('Ticker', group_keys=False)

    # Daily return signals from adjusted close are the main building block.
    df['adj_close_log_return'] = grouped['Adj Close'].transform(lambda s: np.log(s / s.shift(1)))
    df['close_to_close_return'] = grouped['Close'].transform(lambda s: np.log(s / s.shift(1)))
    df['abs_return'] = df['adj_close_log_return'].abs()
    df['squared_return'] = df['adj_close_log_return'] ** 2

    # Range and volume features proxy intraday uncertainty and trading activity.
    safe_volume = df['Volume'].where(df['Volume'] > 0, np.nan)
    safe_high = df['High'].where(df['High'] > 0, np.nan)
    safe_low = df['Low'].where(df['Low'] > 0, np.nan)
    df['log_volume'] = np.log(safe_volume)
    df['high_low_log_range'] = np.log(safe_high / safe_low)
    df['parkinson_var_1d'] = (df['high_low_log_range'] ** 2) / (4 * np.log(2))

    # Rolling features summarize the recent volatility regime over multiple horizons.
    for window in [5, 10, 21, 63]:
        df[f'hist_vol_{window}d'] = grouped['adj_close_log_return'].transform(
            lambda s: s.rolling(window).std() * np.sqrt(252)
        )
        df[f'realized_vol_{window}d'] = grouped['adj_close_log_return'].transform(
            lambda s: np.sqrt(252 / window * s.pow(2).rolling(window).sum())
        )
        df[f'volume_zscore_{window}d'] = grouped['log_volume'].transform(
            lambda s: (s - s.rolling(window).mean()) / s.rolling(window).std()
        )

    # A few extra rolling summaries are useful for nonlinear models later.
    for window in [5, 21]:
        df[f'return_mean_{window}d'] = grouped['adj_close_log_return'].transform(
            lambda s: s.rolling(window).mean()
        )
        df[f'abs_return_mean_{window}d'] = grouped['abs_return'].transform(
            lambda s: s.rolling(window).mean()
        )
        df[f'parkinson_vol_{window}d'] = grouped['parkinson_var_1d'].transform(
            lambda s: np.sqrt(252 / window * s.rolling(window).sum())
        )

    # This is the main supervised-learning target: future 5-day annualized realized vol.
    forward_window = 5
    df['fvol_5d'] = grouped['adj_close_log_return'].transform(
        lambda s: np.sqrt(252 / forward_window * s.shift(-1).pow(2).rolling(forward_window).sum().shift(-(forward_window - 1)))
    )
    df['log_fvol_5d'] = np.log(df['fvol_5d'])
    return df

In [7]:
core_features = add_basic_price_features(core_df)
core_features[['Date', 'Ticker', 'Adj Close', 'adj_close_log_return', 'hist_vol_21d', 'fvol_5d', 'log_fvol_5d']].head(15)

,Date,Ticker,Adj Close,adj_close_log_return,hist_vol_21d,fvol_5d,log_fvol_5d
0,2016-01-04,AAPL,23.730944,NaN,NaN,0.400334,-0.915456
1,2016-01-05,AAPL,23.136267,-0.025378,NaN,0.371848,-0.989270
2,2016-01-06,AAPL,22.683491,-0.019764,NaN,0.390866,-0.939390
3,2016-01-07,AAPL,21.726147,-0.043121,NaN,0.287493,-1.246556
4,2016-01-08,AAPL,21.841030,0.005274,NaN,0.333214,-1.098971
5,2016-01-11,AAPL,22.194681,0.016062,NaN,0.314983,-1.155238
6,2016-01-12,AAPL,22.516806,0.014409,NaN,0.298061,-1.210456
7,2016-01-13,AAPL,21.937893,-0.026047,NaN,0.236530,-1.441681
8,2016-01-14,AAPL,22.417686,0.021635,NaN,0.409392,-0.893081
9,2016-01-15,AAPL,21.879322,-0.024308,NaN,0.396753,-0.924442


## Build market and macro proxy features

Market and macro proxy tables are built separately and then merged into each `(Date, Ticker)` observation. This keeps stock-specific and regime-level information clearly separated.

In [8]:
def make_proxy_features(df, prefix, return_mode='log'):
    """
    Build market / macro proxy features.

    Market ETFs behave like tradable assets, so log returns make sense.
    Rate proxies can go negative, so for them we use level differences instead.
    """
    df = df.sort_values(['Ticker', 'Date']).copy()
    grouped = df.groupby('Ticker', group_keys=False)

    if return_mode == 'log':
        safe_adj = df['Adj Close'].where(df['Adj Close'] > 0, np.nan)
        prev_adj = grouped['Adj Close'].shift(1).where(grouped['Adj Close'].shift(1) > 0, np.nan)
        df['proxy_return'] = np.log(safe_adj / prev_adj)
    elif return_mode == 'diff':
        df['proxy_return'] = grouped['Adj Close'].transform(lambda s: s.diff())
    else:
        raise ValueError("return_mode must be 'log' or 'diff'.")

    # Keep both a range-based uncertainty measure and a normalized level signal.
    safe_high = df['High'].where(df['High'] > 0, np.nan)
    safe_low = df['Low'].where(df['Low'] > 0, np.nan)
    df['proxy_range'] = np.log(safe_high / safe_low)
    df['proxy_level'] = df['Adj Close']

    for window in [5, 21, 63]:
        df[f'proxy_hist_vol_{window}d'] = grouped['proxy_return'].transform(
            lambda s: s.rolling(window).std() * np.sqrt(252)
        )

    for window in [21, 63]:
        df[f'proxy_level_zscore_{window}d'] = grouped['proxy_level'].transform(
            lambda s: (s - s.rolling(window).mean()) / s.rolling(window).std()
        )

    keep = [
        'Date', 'Ticker', 'proxy_return', 'proxy_range', 'proxy_level',
        'proxy_hist_vol_5d', 'proxy_hist_vol_21d', 'proxy_hist_vol_63d',
        'proxy_level_zscore_21d', 'proxy_level_zscore_63d',
    ]
    proxy_df = df[keep].copy()
    proxy_df = proxy_df.rename(columns={'Ticker': f'{prefix}_ticker'})
    return proxy_df


market_proxy_features = make_proxy_features(market_proxy_df, 'market', return_mode='log')
macro_proxy_features = make_proxy_features(macro_proxy_df, 'macro', return_mode='diff')

market_proxy_features.head()

,Date,market_ticker,proxy_return,proxy_range,proxy_level,proxy_hist_vol_5d,proxy_hist_vol_21d,proxy_hist_vol_63d,proxy_level_zscore_21d,proxy_level_zscore_63d
0,2016-01-04,IWM,NaN,0.015916,96.635818,NaN,NaN,NaN,NaN,NaN
1,2016-01-05,IWM,0.002179,0.009358,96.846649,NaN,NaN,NaN,NaN,NaN
2,2016-01-06,IWM,-0.015355,0.014706,95.370911,NaN,NaN,NaN,NaN,NaN
3,2016-01-07,IWM,-0.027074,0.018875,92.823463,NaN,NaN,NaN,NaN,NaN
4,2016-01-08,IWM,-0.017374,0.026935,91.224686,NaN,NaN,NaN,NaN,NaN


In [9]:
def pivot_proxy_features(df, ticker_col, prefix):
    # Convert long-form proxy rows into one wide row per date so they can be
    # merged onto every stock observation from the same day.
    wide = df.pivot(index='Date', columns=ticker_col)
    wide.columns = [f'{prefix}_{ticker}_{feature}' for feature, ticker in wide.columns]
    return wide.reset_index()


market_proxy_wide = pivot_proxy_features(market_proxy_features, 'market_ticker', 'mkt')
macro_proxy_wide = pivot_proxy_features(macro_proxy_features, 'macro_ticker', 'macro')

# Merge stock-level features with same-day market and macro context.
panel_df = core_features.merge(market_proxy_wide, on='Date', how='left')
panel_df = panel_df.merge(macro_proxy_wide, on='Date', how='left')

panel_df.shape

(138270, 107)

## Example modeling panel

At this stage, each row corresponds to a `(Date, Ticker)` pair. The panel contains the target `log_fvol_5d` together with stock-specific, market, and macro features.

In [10]:
# This is the final feature set we will hand to future models.
# It includes stock-specific signals plus same-day market and macro context.
feature_columns = [
    'abs_return', 'squared_return', 'high_low_log_range', 'log_volume',
    'hist_vol_5d', 'hist_vol_10d', 'hist_vol_21d', 'hist_vol_63d',
    'realized_vol_5d', 'realized_vol_10d', 'realized_vol_21d', 'realized_vol_63d',
    'volume_zscore_5d', 'volume_zscore_10d', 'volume_zscore_21d', 'volume_zscore_63d',
    'return_mean_5d', 'return_mean_21d',
    'abs_return_mean_5d', 'abs_return_mean_21d',
    'parkinson_vol_5d', 'parkinson_vol_21d',
]
feature_columns += [col for col in panel_df.columns if col.startswith('mkt_') or col.startswith('macro_')]

# Keep both the raw volatility target and its log transform for later modeling.
model_df = panel_df[['Date', 'Ticker', 'fvol_5d', 'log_fvol_5d'] + feature_columns].dropna().reset_index(drop=True)
print(model_df.shape)
model_df.head()

(133980, 98)


,Date,Ticker,fvol_5d,log_fvol_5d,abs_return,squared_return,high_low_log_range,log_volume,hist_vol_5d,hist_vol_10d,...,macro_HYG_proxy_level_zscore_21d,macro_LQD_proxy_level_zscore_21d,macro_^IRX_proxy_level_zscore_21d,macro_^TNX_proxy_level_zscore_21d,macro_^VIX_proxy_level_zscore_21d,macro_HYG_proxy_level_zscore_63d,macro_LQD_proxy_level_zscore_63d,macro_^IRX_proxy_level_zscore_63d,macro_^TNX_proxy_level_zscore_63d,macro_^VIX_proxy_level_zscore_63d
0,2016-04-05,AAPL,0.197562,-1.621702,0.011859,0.000141,0.011901,18.481915,0.190919,0.183323,...,-0.520611,1.643722,-1.083951,-2.174512,0.023680,1.026073,2.325323,-1.238452,-1.255325,-1.093170
1,2016-04-06,AAPL,0.209733,-1.561921,0.010419,0.000109,0.016169,18.475324,0.164979,0.185560,...,0.654182,1.464421,-0.965764,-1.559862,-0.679652,1.317958,2.151779,-1.206257,-1.040957,-1.358624
2,2016-04-07,AAPL,0.139626,-1.968790,0.022051,0.000486,0.021050,18.661331,0.240736,0.223664,...,-0.221432,1.568103,-0.877832,-2.075475,0.708161,1.099912,2.174090,-1.221767,-1.512423,-0.869497
3,2016-04-08,AAPL,0.200380,-1.607538,0.001105,0.000001,0.014683,18.362276,0.226111,0.220824,...,0.509746,1.275476,-0.841116,-1.527388,0.306475,1.237207,1.950836,-1.303594,-1.260494,-1.016639
4,2016-04-11,AAPL,0.252147,-1.377742,0.003308,0.000011,0.016223,18.583055,0.206184,0.216998,...,0.706467,1.184763,-0.924004,-1.331022,1.222256,1.248646,1.856731,-1.528512,-1.204848,-0.788569


## Save the processed modeling table

The compressed CSV produced here is the main handoff between the data notebook and the modeling notebooks.

In [11]:
model_df.to_csv(MODEL_DF_PATH, index=False, compression='gzip')

model_df_summary = {
    'rows': int(len(model_df)),
    'columns': int(model_df.shape[1]),
    'tickers': int(model_df['Ticker'].nunique()),
    'min_date': str(model_df['Date'].min().date()),
    'max_date': str(model_df['Date'].max().date()),
    'path': str(MODEL_DF_PATH),
}

model_df_summary

{'rows': 133980,
 'columns': 98,
 'tickers': 55,
 'min_date': '2016-04-05',
 'max_date': '2025-12-23',
 'path': '/Users/chonggu/Documents/Mine/Erdos 2026 Spring final project/data/model_df_2016_2025.csv.gz'}